# Negative Prompting and Avoiding Undesired Outputs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/negative-prompting.ipynb)

## Overview
This tutorial explores the concept of negative prompting and techniques for avoiding undesired outputs when working with large language models. We'll use an open-source model (Qwen3 14B Instruct) running locally on Google Colab via HuggingFace Transformers.

## Motivation
As AI language models become more powerful, it's crucial to guide their outputs effectively. Negative prompting allows us to specify what we don't want in the model's responses, helping to refine and control the generated content. This approach is particularly useful when dealing with sensitive topics, ensuring factual accuracy, or maintaining a specific tone or style in the output.

## Key Components
1. Using negative examples to guide the model
2. Specifying exclusions in prompts
3. Implementing constraints via system messages and chat templates
4. Evaluating and refining negative prompts

## Method Details
We'll start by setting up our environment with HuggingFace Transformers and loading a quantized open-source model. Then, we'll explore different techniques for negative prompting:

1. Basic negative examples: We'll demonstrate how to provide examples of undesired outputs to guide the model.
2. Explicit exclusions: We'll use prompts that specifically state what should not be included in the response.
3. Constraint implementation: Using system messages and chat templates, we'll create more complex prompts that enforce specific constraints on the output.
4. Evaluation and refinement: We'll discuss methods to assess the effectiveness of our negative prompts and iteratively improve them.

Throughout the tutorial, we'll use practical examples to illustrate these concepts and provide code snippets for implementation.

## Conclusion
By the end of this tutorial, you'll have a solid understanding of negative prompting techniques and how to apply them to avoid undesired outputs from language models. These skills will enable you to create more controlled, accurate, and appropriate AI-generated content for various applications.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Overview**
- Understand **Motivation**
- Understand **Key Components**
- Understand **Method Details**
- Understand **Conclusion**


## Setup

First, let's install dependencies, load our quantized open-source model, and define a helper function.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import re
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Why "Don't" Doesn't Always Work

Before diving into practical negative prompting techniques, we need to understand a **counterintuitive** aspect of how language models process instructions.

### The Activation Paradox

When you write **"Don't mention cats"**, the model must:

1. **Tokenize** the word "cats" — the embedding for "cats" becomes an active vector in the model's representation.
2. **Attend** to "cats" across every layer — the transformer's self-attention mechanism creates connections between "cats" and every other token in the prompt.
3. **Understand** what "cats" means in context — activating a broad semantic neighborhood (kittens, feline, whiskers, meow, litter box, etc.).
4. **Somehow suppress** all of this activated knowledge during generation.

The problem: steps 1–3 happen automatically and powerfully. Step 4 is fighting against the model's fundamental mechanism. The model doesn't have a "suppress this concept" operation — it only has "predict the next most likely token given everything I've attended to."

This is analogous to the classic psychology experiment: **"Don't think of a white bear."** The instruction itself makes it nearly impossible to comply. In cognitive science, this is called **ironic process theory** (Wegner, 1987) — deliberate attempts to suppress certain thoughts make them *more* likely to surface.

Let's see this in action with an LLM.

In [ ]:
# --- Demonstration: Does "Don't mention X" activate X? ---
# We compare two prompts that should produce similar dog-focused content,
# but one explicitly tells the model NOT to mention cats.

prompt_with_suppression = [
    {"role": "user", "content": "Write a short paragraph about dogs. Do NOT mention cats at all."}
]

prompt_without_suppression = [
    {"role": "user", "content": "Write a short paragraph about dogs."}
]

# Generate multiple samples to see the statistical tendency
NUM_SAMPLES = 3
cat_related_words = ["cat", "cats", "feline", "kitten", "kitty", "meow", "purr"]

print("=" * 70)
print("PROMPT A: 'Write about dogs. Do NOT mention cats.'")
print("=" * 70)
suppressed_cat_counts = []
for i in range(NUM_SAMPLES):
    resp = generate(prompt_with_suppression, max_new_tokens=200)
    count = sum(resp.lower().count(w) for w in cat_related_words)
    suppressed_cat_counts.append(count)
    print(f"\nSample {i+1} (cat-related words found: {count}):")
    print(resp[:300])

print("\n" + "=" * 70)
print("PROMPT B: 'Write about dogs.' (no mention of cats at all)")
print("=" * 70)
clean_cat_counts = []
for i in range(NUM_SAMPLES):
    resp = generate(prompt_without_suppression, max_new_tokens=200)
    count = sum(resp.lower().count(w) for w in cat_related_words)
    clean_cat_counts.append(count)
    print(f"\nSample {i+1} (cat-related words found: {count}):")
    print(resp[:300])

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Avg cat-related words with suppression:    {sum(suppressed_cat_counts)/NUM_SAMPLES:.1f}")
print(f"Avg cat-related words without suppression:  {sum(clean_cat_counts)/NUM_SAMPLES:.1f}")
print("\nThe 'don't mention cats' prompt often introduces the concept it's trying to avoid!")

In [ ]:
# --- Token probability inspection: What does the model predict after "Don't mention"? ---
# This reveals that the model's next-token distribution actively represents
# the forbidden concept — the very act of saying "don't mention X"
# puts X at high probability in the model's predictions.

prompt_text = "Don't mention"
messages_for_logits = [{"role": "user", "content": prompt_text}]
text = tokenizer.apply_chat_template(messages_for_logits, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
    # Get logits for the LAST token position (next-token prediction)
    last_logits = outputs.logits[0, -1, :]
    probs = torch.softmax(last_logits, dim=-1)

# Get top 20 predicted next tokens
top_k = 20
top_probs, top_indices = torch.topk(probs, top_k)

print(f"Prompt: '{prompt_text}'")
print(f"\nTop {top_k} predicted next tokens (what the model 'thinks of'):")
print("-" * 50)
for i in range(top_k):
    token = tokenizer.decode(top_indices[i].item())
    prob = top_probs[i].item()
    print(f"  {i+1:2d}. '{token:15s}'  p = {prob:.4f}")

print("\n→ Notice: the model's top predictions after 'Don't mention' are the very")
print("  concepts you'd want to suppress — the instruction ACTIVATES the concept.")

### Ironic Process Theory in LLMs

The demonstration above illustrates a fundamental limitation: **LLMs don't have a negation operator on attention.** The transformer architecture processes every token in the context window — there is no mechanism to "un-attend" to a concept once it's been mentioned.

When the model sees "Don't mention cats":
- The embedding for "cats" fires, activating a cluster of related representations
- Attention heads connect "cats" to every other token in the sequence
- During generation, the model must actively *work against* these activated pathways
- Sometimes the activation is strong enough to leak through — the model mentions cats anyway, or uses closely related concepts ("unlike their feline counterparts…")

This doesn't mean negative constraints are useless — but it means we need to be **strategic** about how we use them. The rest of this notebook explores when they work, when they don't, and how to build more effective constraints.

## Positive vs Negative Framing: Token Probability Analysis

If negative framing activates the very concepts we want to avoid, what's the alternative? **Positive framing** — telling the model what to DO instead of what NOT to do. Let's measure the difference empirically.

In [ ]:
# --- Compare three framings of the same constraint ---
# Goal: get a jargon-free explanation of neural networks.
#   A) Negative: "Don't use technical jargon"
#   B) Positive: "Use simple, everyday language"
#   C) Combined: Both instructions

topic = "how neural networks learn"

framing_a = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Explain {topic} in two paragraphs. Don't use technical jargon or specialized terminology."}
]

framing_b = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Explain {topic} in two paragraphs. Use simple, everyday language that a 10-year-old could understand."}
]

framing_c = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Explain {topic} in two paragraphs. Use simple, everyday language that a 10-year-old could understand. Avoid technical jargon."}
]

# Jargon words that might appear in a neural network explanation
jargon_words = [
    "gradient", "backpropagation", "activation function", "epoch",
    "loss function", "optimizer", "weight", "bias", "neuron",
    "perceptron", "convolution", "sigmoid", "relu", "softmax",
    "hyperparameter", "regularization", "overfitting", "tensor",
    "stochastic", "inference", "latent", "embedding", "parameter"
]

def count_jargon(text, jargon_list):
    """Count occurrences of jargon words in text."""
    text_lower = text.lower()
    found = {}
    for word in jargon_list:
        c = text_lower.count(word.lower())
        if c > 0:
            found[word] = c
    return found

framings = {
    "A) Negative ('Don't use jargon')": framing_a,
    "B) Positive ('Use simple language')": framing_b,
    "C) Combined (positive + negative)": framing_c,
}

for label, messages in framings.items():
    print("=" * 70)
    print(f"FRAMING: {label}")
    print("=" * 70)
    resp = generate(messages, max_new_tokens=300)
    print(resp[:500])
    found = count_jargon(resp, jargon_words)
    print(f"\n📊 Jargon found ({sum(found.values())} total): {found if found else 'None!'}")
    print()

### Why Positive Framing Works Better

The results above typically show that **positive framing (B)** produces fewer jargon terms than **negative framing (A)**. The combined approach (C) often performs best of all.

The mechanism is straightforward:

| Framing | What the model activates | Effect on generation |
|---------|------------------------|---------------------|
| **Negative**: "Don't use jargon" | Activates representations of *jargon* — the model attends to technical vocabulary | Must fight activated jargon pathways |
| **Positive**: "Use simple language" | Activates representations of *simplicity, clarity, everyday words* | Naturally generates from simple-language distribution |
| **Combined**: Both | Activates simple language AND marks jargon for avoidance | Gets both the positive guidance and the boundary |

**Key insight**: Positive framing works better because it tells the model **what distribution to sample from**, rather than asking it to avoid specific tokens from its default distribution. It's the difference between "walk toward the park" (clear direction) and "don't walk toward the highway" (requires knowing where the highway is and constantly checking).

## Constraint Hierarchy: Where You Place Constraints Matters

In chat-based LLMs, constraints can live in different parts of the prompt: the **system message**, the **user message**, or as a **post-instruction reminder**. Their placement affects how strongly the model attends to them during generation.

In [ ]:
# --- Test constraint placement: system vs user vs post-instruction ---
# Same constraint ("no jargon"), placed at three different positions.

topic = "quantum entanglement"

# Placement A: System message
placement_system = [
    {"role": "system", "content": "You are a helpful assistant. You NEVER use technical jargon or specialized scientific terms. You always explain things using plain everyday language."},
    {"role": "user", "content": f"Explain {topic} in a short paragraph."}
]

# Placement B: In the user message alongside the request
placement_user = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Explain {topic} in a short paragraph. Do not use technical jargon or specialized scientific terms. Use only plain everyday language."}
]

# Placement C: As a post-instruction reminder
placement_post = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Explain {topic} in a short paragraph."},
    {"role": "assistant", "content": "Sure, here's a simple explanation of quantum entanglement:"},
    {"role": "user", "content": "Wait — remember: absolutely no jargon or technical terms. Rewrite using only everyday words."}
]

physics_jargon = [
    "quantum", "particle", "photon", "spin", "superposition",
    "wavefunction", "decoherence", "measurement", "observable",
    "eigenstate", "bohr", "probability amplitude", "entangled state",
    "bell", "nonlocal", "correlation", "collapse"
]

placements = {
    "A) System message": placement_system,
    "B) User message": placement_user,
    "C) Post-instruction reminder": placement_post,
}

for label, messages in placements.items():
    print("=" * 70)
    print(f"PLACEMENT: {label}")
    print("=" * 70)
    resp = generate(messages, max_new_tokens=250)
    print(resp[:400])
    found = count_jargon(resp, physics_jargon)
    print(f"\n📊 Jargon found ({sum(found.values())} total): {found if found else 'None!'}")
    print()

### Why Placement Matters: Attention Distance

The effectiveness of constraint placement relates to how transformer attention works:

- **System message constraints** are processed first and attended to by *every subsequent token* in the sequence. They set a "behavioral prior" for the entire generation. This makes them the strongest position for persistent constraints.
- **User message constraints** are closer to the generation point but compete for attention with the actual request ("explain quantum entanglement"). The model must balance understanding the task with respecting the constraint.
- **Post-instruction reminders** benefit from recency — they're closest to the generation point. But they have less "accumulated attention" than system-level constraints.

**Best practice**: For constraints that must hold throughout the entire response, place them in the **system message**. For one-off constraints specific to a particular request, the **user message** works well. Use **post-instruction reminders** when you need to correct or reinforce after seeing initial results.

⚠️ **Important caveat**: Even system-level constraints can be overridden if the user message strongly activates contradictory patterns. If you ask the model to "explain quantum entanglement" (heavily jargon-laden topic), the system constraint must fight against very strong topical activations.

## When Negative Prompting IS Useful

Despite the activation paradox, negative constraints aren't always counterproductive. They work best when targeting **observable surface features** (format, structure, behavior) rather than deep semantic content. Let's test this.

In [ ]:
# --- Cases where negative constraints work well ---

# Case 1: Format exclusion — "Don't use bullet points"
# Format is a surface feature; the model can easily avoid specific formatting tokens.
print("=" * 70)
print("CASE 1: Format constraint — 'Don't use bullet points'")
print("=" * 70)
msgs_format = [
    {"role": "user", "content": "List 5 benefits of reading. Don't use bullet points or numbered lists. Write in flowing prose."}
]
resp = generate(msgs_format, max_new_tokens=250)
print(resp[:400])
has_bullets = any(line.strip().startswith(('-', '•', '*')) or (len(line.strip()) > 1 and line.strip()[0].isdigit() and line.strip()[1] in '.)')  for line in resp.split('\n') if line.strip())
print(f"\n📊 Contains bullet/numbered list: {has_bullets}")

# Case 2: Behavioral constraint — "Don't ask follow-up questions"
print("\n" + "=" * 70)
print("CASE 2: Behavioral constraint — 'Don't ask follow-up questions'")
print("=" * 70)
msgs_behavior = [
    {"role": "system", "content": "You are a helpful assistant. Never ask follow-up questions. Provide complete answers directly."},
    {"role": "user", "content": "I'm feeling stressed about work."}
]
resp = generate(msgs_behavior, max_new_tokens=250)
print(resp[:400])
has_question = '?' in resp
print(f"\n📊 Contains question marks: {has_question}")

# Case 3: Positive frame + specific negative — "Write formally. Do not use contractions."
print("\n" + "=" * 70)
print("CASE 3: Positive + specific negative — 'Formal, no contractions'")
print("=" * 70)
msgs_combined = [
    {"role": "user", "content": "Write a formal paragraph about climate change. Use a professional, academic tone. Do not use contractions (don't, can't, won't, etc.)."}
]
resp = generate(msgs_combined, max_new_tokens=250)
print(resp[:400])
contractions = ["don't", "can't", "won't", "isn't", "aren't", "it's", "they're", "we're", "couldn't", "shouldn't", "wouldn't", "hasn't", "haven't", "didn't", "doesn't"]
found_contractions = [c for c in contractions if c in resp.lower()]
print(f"\n📊 Contractions found: {found_contractions if found_contractions else 'None!'}")

### The Surface vs Depth Rule

The results above demonstrate a clear pattern:

| Constraint type | Target | Effectiveness | Why |
|----------------|--------|---------------|-----|
| **Format** | Bullet points, lists, headers | ✅ High | Format tokens are distinct and easy to avoid |
| **Behavior** | Questions, apologies, hedging | ✅ High | Behavioral patterns are surface-level routines |
| **Specific tokens** | Contractions, specific words | ✅ High | Individual tokens can be suppressed in the logits |
| **Semantic content** | Concepts, topics, ideas | ⚠️ Low | Requires suppressing deep activations across many related tokens |

**The rule of thumb**: Negative constraints work best for **format and behavior** (observable surface features) and worst for **content and concepts** (deep semantic features). If you can define the constraint in terms of specific tokens or patterns in the output text, negative prompting will likely work. If the constraint is about abstract ideas or knowledge, prefer positive framing.

## Building Robust Constraints

Now let's put it all together. The most effective approach combines **positive framing** (what you want) with **optional negative boundaries** (what to avoid). The pattern is: **"Be X (not Y)"** — lead with the desired behavior, then optionally clarify the boundary.

In [ ]:
# --- Transform 5 common negative constraints into robust positive+negative combos ---

transformations = [
    {
        "label": "1. Simplicity",
        "negative_only": "Explain machine learning. Don't use technical jargon or complex terms.",
        "positive_negative": "Explain machine learning using simple everyday words, as if talking to a curious teenager. Avoid specialized terminology."
    },
    {
        "label": "2. Conciseness",
        "negative_only": "Describe the water cycle. Don't be verbose or include unnecessary details.",
        "positive_negative": "Describe the water cycle in exactly 3 concise sentences. Each sentence should cover one phase. No elaboration needed."
    },
    {
        "label": "3. Objectivity",
        "negative_only": "Write about electric cars. Don't include personal opinions or biased language.",
        "positive_negative": "Write a neutral, fact-based summary of electric cars. Present verifiable claims only, citing both advantages and limitations. Do not editorialize."
    },
    {
        "label": "4. Originality",
        "negative_only": "Write a story opening. Don't use clichés or overused phrases.",
        "positive_negative": "Write a surprising, vivid story opening with fresh imagery and an unexpected first line. Avoid well-worn phrases like 'It was a dark and stormy night'."
    },
    {
        "label": "5. Focus",
        "negative_only": "Explain photosynthesis. Don't go off-topic or include tangential information.",
        "positive_negative": "Explain ONLY the core mechanism of photosynthesis: light in → sugar out. Cover just the inputs, the process, and the outputs in a single focused paragraph."
    },
]

for t in transformations:
    print("=" * 70)
    print(f"TRANSFORMATION: {t['label']}")
    print("=" * 70)

    print("\n❌ Negative-only prompt:")
    msgs_neg = [{"role": "user", "content": t["negative_only"]}]
    resp_neg = generate(msgs_neg, max_new_tokens=200)
    print(resp_neg[:300])

    print("\n✅ Positive+negative prompt:")
    msgs_pos = [{"role": "user", "content": t["positive_negative"]}]
    resp_pos = generate(msgs_pos, max_new_tokens=200)
    print(resp_pos[:300])
    print()

### The Robust Constraint Pattern

From the comparisons above, a clear pattern emerges for building effective constraints:

```
1. LEAD with what you WANT    → Activates the desired output distribution
2. SPECIFY the format/scope   → Narrows the generation space
3. OPTIONALLY add boundaries  → Marks specific things to avoid (surface features)
```

**Examples of the pattern in action:**

| ❌ Negative only | ✅ Positive + Negative |
|-----------------|----------------------|
| "Don't be verbose" | "Be concise — use 3 sentences max" |
| "Don't use jargon" | "Use simple everyday words. Avoid technical terms." |
| "Don't be biased" | "Present facts neutrally with evidence. Don't editorialize." |
| "Don't go off-topic" | "Focus exclusively on X. Cover only inputs, process, outputs." |
| "Don't be boring" | "Open with a vivid, surprising image. Avoid clichés." |

The key insight: **"Be X (not Y)"** is almost always better than **"Don't be Y."** The positive frame gives the model a clear target to aim for, while the negative boundary prevents specific failure modes. Together, they constrain the output from both directions.

## 1. Using Negative Examples

Let's start with a simple example of using negative examples to guide the model's output.

In [ ]:
topic = "photosynthesis"

messages = [
    {"role": "system", "content": "You are a helpful assistant. Do NOT include any of the following in your explanation: "
     "technical jargon or complex terminology; historical background or dates; "
     "comparisons to other related topics. Keep your explanation simple, direct, and focused on the core concept."},
    {"role": "user", "content": f"Provide a brief explanation of {topic}."}
]

response = generate(messages)
print(response)

## 2. Specifying Exclusions

Now, let's explore how to explicitly specify what should be excluded from the response.

In [ ]:
topic = "the benefits of exercise"
exclude = "weight loss or body image"

messages = [
    {"role": "system", "content": f"You are a helpful assistant. Important: Do not mention or reference anything related to {exclude}."},
    {"role": "user", "content": f"Write a short paragraph about {topic}."}
]

response = generate(messages)
print(response)

## 3. Implementing Constraints

Let's use system messages with explicit constraints to enforce specific rules on the model's output.

In [ ]:
topic = "artificial intelligence"
style = "technical"
excluded_words = "robot, human-like, science fiction"

messages = [
    {"role": "system", "content": f"You are a helpful assistant. Write a {style} description. "
     f"Constraints: 1) Do not use any of these words: {excluded_words}. "
     "2) Keep the description under 100 words. "
     "3) Do not use analogies or metaphors. "
     "4) Focus only on factual information."},
    {"role": "user", "content": f"Describe {topic}."}
]

response = generate(messages)
print(response)

## 4. Evaluation and Refinement

To evaluate and refine our negative prompts, we can create a function that checks if the output adheres to our constraints.

In [ ]:
def evaluate_output(output, constraints):
    """Evaluate if the output meets the given constraints."""
    results = {}
    for constraint, check_func in constraints.items():
        results[constraint] = check_func(output)
    return results

# Define some example constraints
constraints = {
    "word_count": lambda x: len(x.split()) <= 100,
    "no_excluded_words": lambda x: all(word not in x.lower() for word in ["robot", "human-like", "science fiction"]),
    "no_analogies": lambda x: not re.search(r"\b(as|like)\b", x, re.IGNORECASE)
}

# Evaluate the previous output
evaluation_results = evaluate_output(response, constraints)
print("Evaluation results:", evaluation_results)

# If the output doesn't meet all constraints, we can refine our prompt
if not all(evaluation_results.values()):
    refined_style = "technical and concise"
    refined_excluded = "robot, human-like, science fiction, like, as"

    refined_messages = [
        {"role": "system", "content": f"You are a helpful assistant. Write a {refined_style} description. "
         f"Constraints: 1) Do not use any of these words: {refined_excluded}. "
         "2) Keep the description under 100 words. "
         "3) Do not use analogies or metaphors. "
         "4) Focus only on factual information."},
        {"role": "user", "content": "Describe artificial intelligence."}
    ]

    refined_response = generate(refined_messages)
    print("\nRefined response:\n", refined_response)

    # Evaluate the refined output
    refined_evaluation = evaluate_output(refined_response, constraints)
    print("\nRefined evaluation results:", refined_evaluation)

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** rewrite a prompt using the technique from this notebook. Document what changes and why.

**Exercise 2 — Build:** test the technique on a different task and compare results. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** combine this technique with one from a previous notebook.

## 📝 Key Takeaways

- **Overview** — revisit this section for reference
- **Motivation** — revisit this section for reference
- **Key Components** — revisit this section for reference
- **Method Details** — revisit this section for reference
- **Conclusion** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the prompt-engineering/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [Multilingual-Prompting](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/multilingual-prompting.ipynb) | ➡️ **Next:** [Prompt-Chaining-Sequencing](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/prompt-chaining-sequencing.ipynb)